# 02: Baseline models

Trains Logistic Regression, Random Forest and XGBoost on the Telco data and compares them. All logic lives in `src/ai_experiment_lab/`; this notebook only calls it.

**Selection** uses cross-validation on the training split. The test split is evaluated once per model and is never used to choose between models.

Run order: setup cell, then each cell below.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/<your-username>/ai-experiment-lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
from pathlib import Path
import pandas as pd

from ai_experiment_lab.data import load_csv, validate_dataframe, validation_thresholds
from ai_experiment_lab.preprocessing import split_data
from ai_experiment_lab.training import run_baselines
from ai_experiment_lab.utils.config import get_data_dir, load_config

cfg = load_config(f"{REPO_DIR}/configs/default.yaml")
models_cfg = load_config(f"{REPO_DIR}/configs/model_configs.yaml")["classification"]
d, seed = cfg["dataset"], cfg["project"]["random_seed"]

df, info = load_csv(Path(get_data_dir()) / d["file"])
print(info.name, info.n_rows, "rows | sha256", info.sha256[:12])

result = validate_dataframe(df, d["target"], d["task"], validation_thresholds(cfg), d["id_columns"])
print(result.report())
result.raise_if_rejected()

In [ ]:
sp = cfg["split"]
split = split_data(df, d["target"], d["positive_label"], d["id_columns"],
                   sp["test_size"], seed, sp["stratify"])
print("train:", split.X_train.shape, " test:", split.X_test.shape)
print("churn rate train / test: %.4f / %.4f" % (split.y_train.mean(), split.y_test.mean()))

In [ ]:
res = run_baselines(split, models_cfg, cfg["preprocessing"], seed=seed, cv_folds=5,
                    primary_metric=cfg["selection"]["primary_metric"])
pd.set_option("display.width", 200)
cols = ["model", "cv_f1", "cv_primary_std", "cv_pr_auc", "test_f1", "test_recall",
        "test_precision", "test_roc_auc", "test_pr_auc", "test_accuracy", "fit_seconds"]
res.table[cols].round(4)

In [ ]:
for name, c in res.confusion.items():
    print(f"{name:22s} TN={c['tn']:4d} FP={c['fp']:4d} FN={c['fn']:4d} TP={c['tp']:4d}")

In [ ]:
# Save the table to Drive so results survive the session (MLflow replaces this in Phase 2).
out = Path(DRIVE_ROOT) / "results"
out.mkdir(exist_ok=True)
res.table.to_csv(out / "baseline_results.csv", index=False)
print("saved", out / "baseline_results.csv")

## Record in `docs/experiments/EXP-001-baselines.md`

Copy the table above into the experiment report, then write the conclusion: which model won on the cross-validation score, how close the test score is to the cross-validation score, and why you think one model beat another.